In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
===========================================================================
Vehicle Diagnostic Data Processing Pipeline
===========================================================================

Author      : Sanmathi S
Technology  : Python, AWS S3, Pandas, XML Processing
Version     : Portfolio Edition 1.0

Project Overview
----------------
This solution automates the processing of vehicle diagnostic scan logs
stored in AWS S3 and converts unstructured XML data into structured,
analytics-ready datasets.

The pipeline downloads diagnostic files, extracts vehicle information,
calculates diagnostic trouble code (DTC) statistics, filters valid VINs,
handles duplicate records, and generates consolidated output reports.

Key Features
------------
• AWS S3 Integration
• XML Scan Log Processing
• VIN-Based Analysis
• ECU Classification
• Active DTC Calculation
• Inactive DTC Calculation
• Duplicate Record Removal
• Batch Processing Framework
• Error Tracking & Logging
• Automated CSV Generation
• Analytics Dataset Creation

Input
-----
Source:
• AWS S3 Storage

File Type:
• XML Diagnostic Scan Logs

Output
------
• Processed Diagnostic Dataset
• VIN Validation Reports
• Analytics Data Files
• Error Log Reports

Business Benefits
-----------------
• Eliminates manual file processing
• Handles large-scale diagnostic datasets
• Improves data quality
• Accelerates diagnostics analysis
• Enables analytics-ready reporting

Domain
------
Automotive Diagnostics | Vehicle Telematics | Data Engineering

Confidentiality Notice
----------------------
This portfolio version has been anonymized.

The following production information has been removed:

• AWS Credentials
• Client Names
• Bucket Names
• Internal Paths
• Vehicle Identifiers
• User Information
• Organization-Specific References

Sample values and placeholders are used for public demonstration purposes.

===========================================================================
"""


import os
import boto3
import xmltodict
import pandas as pd
import numpy as np
from datetime import datetime
import traceback
from tqdm import tqdm
from collections import defaultdict
import json
import warnings
import pickle as pkl
 
def download_file_to_path(file_name, folder_name):
    client=boto3.client('s3', aws_access_key_id='YOUR_ACCESS_KEY', aws_secret_access_key='YOUR_SECRET_KEY')
    local=os.path.basename(file_name)
    os.makedirs(os.path.join(os.getcwd(),folder_name),exist_ok=True)
    client.download_file('your_s3_bucket',file_name,os.path.join(os.getcwd(),folder_name,local))
    
    
# Define a function to parse the dates
def parse_date(date_str):
    try:
        return parser.parse(date_str)
    except ValueError:
        return pd.NaT

label_dict={'Engine ECU':[
    'BOSCH - BSVI',
    'BOSCH - BSVI IFX',
    'Denso A4 - BSVI',
    'Denso H6-2V - BSVI',
    'Denso A6 - BSVI',
    'DELPHI_DINEX - BSVI',
    'DELPHI - BSVI',
    'Denso H6-4V - BSVI',
    'CNG WW - BSVI',
    'BOSCH - BSVI IFX V2',
    'BOSCH_ZD_30 - BSVI',
    'Advantek A46-H4NA - BSVI'
    ],
    'ACU':
    [
        'Cummins H4 - BSVI',
        'Cummins H6_N4 - BSVI',
        'Albonair-H6-2V_A6 - BSVI',
        'Albonair-H6-4V - BSVI',
        'BOSCH_DINEX - BSVI',
        'DINEX_PO_DCU - BSVI',
    ]}
 
def first_record(path):
    s3 =  boto3.client('s3', aws_access_key_id='YOUR_ACCESS_KEY', aws_secret_access_key='YOUR_SECRET_KEY')
    file=s3.get_object(Bucket=bucket_name,Key=path)['Body']
    data=xmltodict.parse(file.read())
    # print(data['DiagnosticScanLog']['Header']['LogStartDateTime'])
    if 'AM' in data['DiagnosticScanLog']['Header']['LogStartDateTime'] or 'PM' in data['DiagnosticScanLog']['Header']['LogStartDateTime']:
        reference_date=datetime.strptime(data['DiagnosticScanLog']['Header']['LogStartDateTime'],'%d-%m-%Y %I:%M:%S %p')
    else:
        reference_date=datetime.strptime(data['DiagnosticScanLog']['Header']['LogStartDateTime'],'%d-%m-%Y %H:%M:%S')
    ref=pd.DataFrame(data['DiagnosticScanLog']['DiagnosticScanData']['FailureInformationData']['Event'])
    cols=[k for k,v in data['DiagnosticScanLog']['Header'].items() if v!=None]
    for k in cols:
        ref[k]=data['DiagnosticScanLog']['Header'][k]
    ref=ref[cols+['EventType', 'Logdate', 'Logtime', 'VehType', 'ActiveDTCs', 'ECU']].copy()
    ref['Logdate']=[reference_date.strftime('%d-%m-%Y')]*len(ref)
    ref['Logdate']=pd.to_datetime(ref['Logdate'],format="%d-%m-%Y")
    ref['Logtime']=ref['Logtime'].apply(lambda x: x.replace('.',':'))
    ref['Logtime']=pd.to_datetime(ref['Logtime'],format="%H:%M:%S:%f")
    # Update LogStartDateTime by combining date from 'LogStartDateTime' and time from 'Logtime'
    ref['LogStartDateTime'] = pd.to_datetime(ref['LogStartDateTime'].apply(lambda x: x.split()[0]) + ' ' + ref['Logtime'].astype(str))

    # Apply the parse_date function to the LogStartDateTime column
    ref['LogStartDateTime'] = ref['LogStartDateTime'].apply(parse_date)
 
    # Convert LogStartDateTime to a datetime object
    ref['LogStartDateTime'] = pd.to_datetime(ref['LogStartDateTime'])
 
    # Extract the date part from LogStartDateTime
    ref['Date'] = ref['LogStartDateTime']
 
    # Sort by LogStartDateTime to ensure the earliest times are first
    ref = ref.sort_values(by='LogStartDateTime')
 
    # Convert LogStartDateTime to a string format of "dd-mm-yy"
    ref['LogStartDateTime'] = ref['LogStartDateTime'].dt.strftime('%d-%m-%y')
 
    # Identify duplicate records
    duplicates = ref[ref.duplicated(subset=['VIN', 'Date'], keep=False)]
 
    # Drop duplicates, keeping only the first occurrence for each VIN and LogDate
    ref = ref.loc[ref.groupby(['VIN', 'Date'])['LogStartDateTime'].idxmin()]
 
 
    event_list=ref['EventType'].tolist()
    #if erase dtc event is present then take the time
    if 'Erase DTCs' in event_list:
        erase_dtc_time=min(ref[ref['EventType']=='Erase DTCs']['LogStartDateTime'].tolist())
    else:
        erase_dtc_time=np.nan

    #first instance of the day
    first_instance=pd.DataFrame(ref.iloc[ref['LogStartDateTime'].idxmin()]).T
    if len(first_instance.loc[0,'VIN'].strip())!=17:
        return None
    first_instance['ECU']=first_instance['ECU'].apply(lambda x: [x] if type(x)!=list else x)
    first_instance=first_instance.explode('ECU')
    first_instance['ECUName']=first_instance['ECU'].apply(lambda x: x['ECUName'] if type(x)==dict else x)
    first_instance=first_instance.merge(first_instance['ECU'].apply(pd.Series),on='ECUName')
 
    def count_active_inactive(x):
        if type(x)==dict and ('DTC' in x.keys()):
            active_count=0
            inactive_count=0
            if type(x['DTC'])==list:
                for vals in x['DTC']:
                    if vals['Status']=='Inactive':
                        inactive_count+=1
                    else:
                        active_count+=1
            elif type(x['DTC'])==dict:
                if x['DTC']['Status']=='Inactive':
                    inactive_count+=1
                else:
                    active_count+=1
            else:
                pass
 
            return active_count,inactive_count
        else:
            return np.nan,np.nan
 
    first_instance['DTC_Count']=first_instance['DTCList'].apply(lambda x: len(x['DTC']) if type(x)==dict else 0)
    # Calculate active and inactive DTCs
    dtc_counts = first_instance['DTCList'].apply(count_active_inactive)
 
    # Assign them to the respective columns
    first_instance['ActiveDTC_Count'] = dtc_counts.apply(lambda x: x[0])
    first_instance['InactiveDTC_Count'] = dtc_counts.apply(lambda x: x[1])
    first_instance=first_instance.reset_index(drop=True)
    #ECUType from label and ECUName label dict schema : {ECUType:[ECUName1,ECUName2,ECUName3]}
    first_instance['ECUType']=first_instance['ECUName'].apply(lambda x: [k for k,v in label_dict.items() if x in v][0] if x in [j for i in label_dict.values() for j in i] else 'Others')
 
    

    op=first_instance[['SessionName', 'UserName', 'VIN','LogStartDateTime','ECUType' ,'ActiveDTC_Count','InactiveDTC_Count','DTC_Count']]
    #pivot ecutype and count of active and inactive dtc
    op=op.pivot_table(index=['SessionName', 'UserName', 'VIN',
        'LogStartDateTime'],columns='ECUType',values=['ActiveDTC_Count',
        'InactiveDTC_Count'],aggfunc='sum').reset_index()
 
    op['EraseDTC_Time']=erase_dtc_time
    op['Events']=','.join(event_list)
    #convert erase dtc time and LogStartDateTime to human readable datetime
    op['EraseDTC_Time']=pd.to_datetime(op['EraseDTC_Time'])
    op['LogStartDateTime']=pd.to_datetime(op['LogStartDateTime'])
    op['LogStartDateTime']=op['LogStartDateTime'].dt.strftime('%d-%m-%Y %H:%M:%S')
    # print(op['LogStartDateTime'])
    op['EraseDTC_Time']=op['EraseDTC_Time'].dt.strftime('%d-%m-%Y %H:%M:%S')
    op.loc[:,'filename']=str(path)
    del data
    return op
  #merge multi index columns
def process_op(df):
    col_list=df.columns
    df.columns=[': '.join(col) for col in col_list]
    return df
first_instance_list=[]
successfull_files=[]
error_files=[]
error_code=defaultdict(list)
 
def s3_walk(bucket_name, start_prefix='', s3_client=None):
    """
    A generator that walks through an S3 bucket's 'directories' and 'files',
    similar to os.walk. Yields a tuple (current_prefix, subprefixes, objects).
 
    :param bucket_name: Name of the S3 bucket.
    :param start_prefix: Prefix to start listing from (acts like a directory path).
    :param s3_client: An instance of boto3.client('s3'). If None, will be created.
    """
    if s3_client is None:
        s3_client = boto3.client('s3', aws_access_key_id='YOUR_ACCESS_KEY', aws_secret_access_key='YOUR_SECRET_KEY')
    paginator = s3_client.get_paginator('list_objects_v2')
 
    # List objects and prefixes (subdirectories) under the current prefix
    operation_parameters = {'Bucket': bucket_name, 'Prefix': start_prefix, 'Delimiter': '/'}
 
    for page in paginator.paginate(**operation_parameters):
        # Extract subdirectories (common prefixes)
        subprefixes = [prefix['Prefix'] for prefix in page.get('CommonPrefixes', [])]
 
        # Extract actual file objects
        objects = [obj['Key'] for obj in page.get('Contents', []) if obj['Key'] != start_prefix]
 
        # Yield current 'directory' structure
        yield (start_prefix, subprefixes, objects)
 
        # Recursively list in each subdirectory
        for subprefix in subprefixes:
            yield from s3_walk(bucket_name, subprefix, s3_client)
 
 
try:
    path_path=os.path.join(os.getcwd(),'1path_list_novdec_test.pkl')
    with open(path_path, 'rb') as file:
        files = pkl.load(file)
except Exception as e:
    files=[]
    for current_prefix, subprefixes, objects in tqdm(s3_walk('connectallfiles')):
        files.extend(objects)
 
    path_path=os.path.join(os.getcwd(),'1path_list_novdec_test.pkl')
 
    with open(path_path, 'wb') as file:
        pkl.dump(files, file)

files=[path for path in files if 'sample_month' in path] #for april data use s3 folder name for april month , contact madan for s3 cloud folder name
paths=[slg for slg in files if slg.endswith('.slg')] #list of all the files
os.makedirs('processed_batch_files',exist_ok=True)
completion_path=os.path.join(os.getcwd(),'processed_batch_files') #replace revision_1 with april
file_path=os.path.join(os.getcwd(),'processed_batch_files') #replace revision_1 with april
df_list=[pd.read_csv(os.path.join(file_path,file)) for file in os.listdir(completion_path) if file.endswith('.csv')]
completed_file_list=[]
for df in df_list:
    completed_file_list.extend(df['filename: '].tolist())
bucket_name = 'yyyyy'
remaining_paths=list(set(paths)-set(completed_file_list))
prod_path=remaining_paths
print("bucket_name:", bucket_name)
print("bucket_name len:", len(bucket_name))
print("Number of prod_path:", len(prod_path))
print("Number of completed_file_list:", len(completed_file_list))
print("Number of completion_path:", len(completion_path))
print("Number of paths:", len(paths))
print("Number of files:", len(files))
print("remaining_paths:", len(remaining_paths))
print("file_path:", file_path)
print("Number of df_list:", len(df_list))
# print("df_list:",df_list)
len(prod_path)
 
 
# batches=[prod_path[i:i+1000] for i in range(0,len(prod_path),1000)]
#sort batches
 
batches = [prod_path[i:i+1000] for i in range(0, min(1000, len(prod_path)), 1000)]
 
# ... rest of the code remains the same ...
warnings.filterwarnings("ignore")
#ignore  UserWarning
 
print("Number of files in this batch:", len(batches))
# print("Files in this batch:", batches)
 
 
# Initialize error_dict
error_dict = {}
 
try:
    unsorted_val = [int(batch.split('_')[1][:-4]) for batch in os.listdir(completion_path)]
    sorted_val = max(unsorted_val)
except Exception as e:
    sorted_val = -1
 
def print_dates(file):
    name = file
    s3 = boto3.client('s3', aws_access_key_id='YOUR_ACCESS_KEY', aws_secret_access_key='YOUR_SECRET_KEY')
    file = s3.get_object(Bucket=bucket_name, Key=path)['Body']
    data = xmltodict.parse(file.read())
    reference_date = data['DiagnosticScanLog']['Header']['LogStartDateTime']
    print(f'Header stamp:{reference_date} for file {name}')
    ref = pd.DataFrame(data['DiagnosticScanLog']['DiagnosticScanData']['FailureInformationData']['Event'])
    cols = [k for k, v in data['DiagnosticScanLog']['Header'].items() if v != None]
    for k in cols:
        ref[k] = data['DiagnosticScanLog']['Header'][k]
    ref = ref[cols + ['EventType', 'Logdate', 'Logtime', 'VehType', 'ActiveDTCs', 'ECU']].copy()
    reference_date = reference_date.replace('/', '-').replace('\', '-')
    ref['Logdate'] = [reference_date.strftime('%d-%m-%Y')] * len(ref)
    try:
        ref['Logdate'] = pd.to_datetime(ref['Logdate'], format="%d-%m-%Y")
    except ValueError:
        ref['Logdate'] = pd.to_datetime(ref['Logdate'], format="%m-%d-%Y")
    ref['Logtime'] = ref['Logtime'].apply(lambda x: x.replace('.', ':'))
    ref['Logtime'] = pd.to_datetime(ref['Logtime'], format="%H:%M:%S:%f")
    print(ref['Logdate'].min())
    print(ref['Logtime'].min())
 
new_batch = sorted_val + 1
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
 
for batch_no, batch in tqdm(enumerate(batches), leave=False):
    batch_no = batch_no + new_batch
    first_instance_list = []
    for path in batch:
        try:
            first_instance_list.append(first_record(path))
            successfull_files.append(path)
        except Exception as e:
            if type(e) == ValueError:
                try:
                    print_dates(path)
                except Exception:
                    error_dict[path] = traceback.format_exc()
                    with open('error12.json', 'w') as file:
                        json.dump(error_dict, file)
                    with open('error1.txt', 'a') as errors:
                        errors.write(f'{path} with error {e} \n')
            else:
                error_dict[path] = traceback.format_exc()
                with open('error12.json', 'w') as file:
                    json.dump(error_dict, file)
                with open('error1.txt', 'a') as errors:
                    errors.write(f'{path} with error {e} \n')
            continue
 
    first_instance_list = [df for df in first_instance_list if type(df) != type(None)]
    if len(first_instance_list) > 0:
        phase_2 = pd.concat([process_op(sample) for sample in first_instance_list], axis=0, ignore_index=True)
        phase_2 = phase_2[
            ['SessionName: ', 'filename: ', 'UserName: ', 'VIN: ', 'LogStartDateTime: ', 'ActiveDTC_Count: Engine ECU', 'ActiveDTC_Count: ACU',
             'ActiveDTC_Count: Others', 'InactiveDTC_Count: Engine ECU', 'InactiveDTC_Count: ACU', 'InactiveDTC_Count: Others', 'EraseDTC_Time: ']]
        phase_2['ActiveDTC_Count: Total'] = phase_2[
            ['ActiveDTC_Count: Engine ECU', 'ActiveDTC_Count: ACU', 'ActiveDTC_Count: Others']].sum(axis=1)
        phase_2['InactiveDTC_Count: Total'] = phase_2[
            ['InactiveDTC_Count: Engine ECU', 'InactiveDTC_Count: ACU', 'InactiveDTC_Count: Others']].sum(axis=1)
        os.makedirs('rectified', exist_ok=True)
 
        print(os.getcwd(), 'processed_batch_files' + f"rows={phase_2.shape[0]}" + f"columns={phase_2.shape[1]}")
        phase_2.to_csv(os.path.join(os.getcwd(), 'processed_batch_files', f'batch_{batch_no}.csv'), index=False)
        # del phase_2
 
    print("Processing batch number", batch_no)

 
    
# number of successfull and error files
print(f"Number of successfull files:{len(successfull_files)}")
print(f"Number of error files:{len(error_files)}")
# save error files to csv
# save to text file
with open('error_files.txt','w') as file:
    file.write(f"Number of successfull files:{len(successfull_files)} \n Number of error files:{len(error_files)}")
    file.close()
 
csvs = [os.path.join(os.getcwd(), 'processed_batch_files', csv) for csv in os.listdir(os.path.join(os.getcwd(),'processed_batch_files')) if csv.startswith('batch_')]
print("CSV files:", csvs)
print("Directory contents:", os.listdir(os.path.join(os.getcwd(),'processed_batch_files')))
compiled = pd.concat([pd.read_csv(csv) for csv in csvs], axis=0, ignore_index=True)
len(paths)
compiled.shape
compiled.to_csv('processed_batch_files.csv')
 
file_path = os.path.join(os.getcwd(), 'processed_batch_files')
df_list = [pd.read_csv(os.path.join(file_path, file)) for file in os.listdir(file_path) if os.path.isfile(os.path.join(file_path, file)) and file.endswith('.csv')]
 
compiled=pd.concat(df_list,axis=0, ignore_index=True)
compiled.shape
compiled
 
unique=compiled.drop_duplicates(subset='filename: ')
unique.shape
unique
 
with open('usernames.txt', 'r') as f:
    username_filter = [line.strip().lower() for line in f.readlines()]
 
with_username = unique[unique['UserName: '].str.lower().isin(username_filter)]
without_username = unique[~unique['UserName: '].str.lower().isin(username_filter)]
 
print('with_username', with_username.shape)
print('without_username', without_username.shape)
 
with_username.to_csv('15WithUsernames2.csv', index=False)
without_username.to_csv('15WithoutUsernames2.csv', index=False)
 
startswith_m=without_username[without_username['VIN: '].apply(lambda x:x.startswith('M'))]
print('M as first character files:',startswith_m.shape)
 
length_filter=startswith_m[startswith_m['VIN: '].apply(len)==17]
print('VIN Start with M &  17 characters & Without username:', length_filter.shape)
filtered_vin_list=length_filter
 
filtered_vin_list
 
filtered_vin_list.to_csv('filtered_vehicle_records.csv')
 
 
compiled.to_pickle("15Sscanlognovdec_test.pkl")  
 
 
print(f"Number of successfull files:{len(successfull_files)}")
print(f"Number of error files:{len(error_files)}")
print('without_username',without_username.shape)
print('with_username',with_username.shape)
print('VIN Start with M &  17 characters & Without username:', length_filter.shape)
print(f"Number of unique records with all the validatoion :{len(filtered_vin_list)}")